# Local ROM Stability Check

Build a FOM trajectory from a chosen initial condition, construct POD/DEIM ROMs, run the true-nonlinearity ROM, and inspect capture/trajectory/modes before training a neural ROM.

In [ ]:
using Plots

### ADJUSTED: Use shared ROM stability helpers instead of notebook-local function definitions.
include(joinpath(@__DIR__, "..", "..", "src", "HPC", "Tools", "ROM_stability_helpers.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Visualizations", "optimization_visualizations.jl"))


## Parameters

In [ ]:
N = 256
L = 1.0
ε2 = 1e-2
k = 1.0
tspan = (0.0, 2.0)
reference_dt_factor = 0.5
dimension = 2
boundary_condition = "periodic"
N_obs = 100
h = 8
seed = 1

r = 4
m = 4

interface_width = sqrt(2ε2)
initial_condition_examples = [
    (name="default", u0=nothing),

    (name="2d circle drop", u0=(x, y) -> tanh((sqrt((x - 0.5)^2 + (y - 0.5)^2) - 0.23) / interface_width)),
    (name="2d offcenter drop", u0=(x, y) -> tanh((sqrt((x - 0.35)^2 + (y - 0.58)^2) - 0.18) / interface_width)),

    
    (name="2d one-direction tanh front", u0=(x, y) -> tanh((x - 0.5) / interface_width)),
   
    (name="2d annulus", u0=(x, y) -> max(
        tanh((sqrt((x - 0.5)^2 + (y - 0.5)^2) - 0.30) / interface_width),
        tanh((0.15 - sqrt((x - 0.5)^2 + (y - 0.5)^2)) / interface_width),
    )),
    (name="2d soft bleedout patch and static slab", u0=(x, y) ->
    0.5 * (1 - tanh((x - 0.50) / (2.0interface_width))) *
        (0.55sin(6π * x / L) * sin(4π * y / L) +
         0.20sin(10π * x / L + 0.7) * sin(2π * y / L)) +
    0.5 * (1 + tanh((x - 0.50) / (2.0interface_width))) *
        tanh((x - 0.72) / (1.2interface_width))
),
    (name="2d sin xy", u0=(x, y) -> sin(2π * x / L) * sin(2π * y / L)),
    (name="2d high frequency x sine", u0=(x, y) -> sin(3π * x / L)),
    ];

initial_condition_2d_examples = filter(ic -> startswith(ic.name, "2d "), initial_condition_examples)


## Functions

## Initial Conditions

In [ ]:
### ADJUSTED: Avoid clobbering the Allen-Cahn parameter k while previewing initial conditions.
for ic_index in 1:length(initial_condition_2d_examples)
    println("k=$ic_index")
    selected_initial_condition_2d_index = ic_index
    selected_initial_condition = initial_condition_2d_examples[selected_initial_condition_2d_index]
    display(
        ac_plot_stability_initial_conditions([selected_initial_condition];
         N, L, ε2, dimension, boundary_condition, show_colorbar = true)
         )
end

## Run FOM, Build ROM, Run ROM

In [ ]:
selected_initial_condition = initial_condition_2d_examples[7]

u0 = selected_initial_condition.u0
fom = ac_run_stability_fom_reference(; N, L, ε2, k, tspan, reference_dt_factor, u0, dimension, boundary_condition)
println("Ran FOM reference")
rom = ac_build_stability_rom(fom, r, m; N_obs, h, seed)
println("Built ROM")
rom_run = ac_run_stability_rom(fom, rom);
println("Ran ROM ")


## Singular-Value Capture

In [ ]:
# stability_capture_table(fom, ac_build_stability_rom; rs=[2, 4, 8, 10, 15, 20], ms=[2, 4, 8, 10, 15, 20])

## Trajectory

In [ ]:
trajectory_gifs(fom, rom_run; out_dir=joinpath(@__DIR__, "rom_stability_gifs"), max_frames=15, fps=1, show_colorbar = true)

In [ ]:
true_u = hcat(fom.u_ref.u...)
rom_u = rom_run.reconstructed

true_change = [maximum(abs.(true_u[:, j] .- true_u[:, 1])) for j in axes(true_u, 2)]
rom_change = [maximum(abs.(rom_u[:, j] .- rom_u[:, 1])) for j in axes(rom_u, 2)]

@show size(true_u)
@show size(rom_u)
@show length(fom.u_ref.t)
@show extrema(true_u)
@show extrema(rom_u)
@show maximum(true_change)
@show maximum(rom_change)
@show true_change[1:10:end]

## Modes

In [ ]:
plot_rom_modes(fom, rom, n_state=10, n_deim=10)